# 3. Data Preparation

Terceira fase do CRISP-DM. O que entra aqui é o CSV bruto carregado direto de `data/raw/`; o que sai é um dataset processado, pronto para a EDA do notebook 04. Esta fase é menos sobre análise e mais sobre decisões de representação: como categorizar a target, que tipos de coluna usar, se vamos transformar variáveis assimétricas, e como garantir que tudo seja reprodutível.

A regra que sigo aqui é "transformações idempotentes e auditáveis": cada passo é determinístico, e o dataset processado pode ser regenerado a qualquer hora rodando o notebook do zero.

## 3.1 Carregamento e setup

A partir desta fase, o mesmo dataset vai aparecer em quatro notebooks seguidos (03 prep, 04 EDA, 05 regressão, 06 classificação). Para evitar duplicar a função de carregamento em cada um, criei um módulo reutilizável em `src/nps/data.py` com a função `carregar_dataset_bruto()`.

### Sobre como importar código de `src/` no notebook

Existem dois caminhos comuns para um notebook usar um módulo Python que mora em outro diretório do projeto. Vale entender os dois antes de escolher um.

**Caminho 1: pacote instalado em modo editável (o que vamos usar)**

```python
from nps.data import carregar_dataset_bruto
```

Funciona porque o `pyproject.toml` declara `src/nps` como um pacote empacotável (via hatch), e o `uv sync` instala esse pacote dentro da virtualenv apontando para o código-fonte real (modo editável). Resultado: qualquer arquivo dentro do `.venv` consegue importar `nps` como se fosse uma biblioteca de terceiro, e mudanças no código de `src/nps/` aparecem imediatamente.

**Caminho 2: manipulação de `sys.path` no notebook**

```python
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from nps.data import carregar_dataset_bruto
```

Essa é a forma mais explícita: o notebook diz literalmente "adicione `../src` à lista de lugares onde Python procura módulos". É útil em projetos sem `pyproject.toml` ou em ambientes restritos onde não se pode instalar pacote local.

### Tradeoffs entre os dois caminhos

| Aspecto | Pacote instalado | sys.path |
|---|---|---|
| Verbosidade no notebook | Mínima (uma linha) | 3-4 linhas de boilerplate |
| Setup inicial | Exige `pyproject.toml` correto e `uv sync` | Funciona sem nada extra |
| Funciona em scripts e testes | Sim | Só onde o `sys.path` foi configurado |
| Ferramentas de IDE (autocomplete, type-checker) | Funcionam bem | Geralmente confusas |
| Risco de "import quebrado" ao mexer em paths | Baixo | Alto (depende do `cwd`) |
| Prática profissional | Padrão da indústria | Comum em didática e scripts ad-hoc |

Vou seguir com o caminho 1 porque o projeto já está configurado para isso, e porque toda evolução adiante (testes, pipeline, scripts) vai se beneficiar do mesmo import limpo. O caminho 2 fica como referência para quando aparecer um projeto onde não dá para instalar pacote local.

In [1]:
# Bibliotecas principais
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Funcao do nosso modulo (definida em src/nps/data.py)
from nps.data import carregar_dataset_bruto

# Reprodutibilidade
SEMENTE_ALEATORIA = 42
np.random.seed(SEMENTE_ALEATORIA)

# Configuracao visual padrao
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

# Caminhos para esse notebook (figuras desta fase)
RAIZ_PROJETO = Path.cwd().parent
PASTA_FIGURAS = RAIZ_PROJETO / "reports" / "figures" / "03_data_preparation"
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Pasta de saida do dataset processado
PASTA_DADOS_PROCESSADOS = RAIZ_PROJETO / "data" / "processed"
PASTA_DADOS_PROCESSADOS.mkdir(parents=True, exist_ok=True)

# Carregamento do dataset bruto via funcao do modulo
dados = carregar_dataset_bruto()

print(f"Dataset carregado: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas")

Dataset carregado: 2,500 linhas x 19 colunas


A partir daqui, `dados` contém o CSV bruto exatamente como está em `data/raw/`. As próximas seções vão derivar variáveis novas e gerar o dataset processado em `data/processed/`. Importante: as transformações que fizermos aqui **não modificam** `data/raw/`. Em projetos de DS, o diretório `raw/` deve ser sempre tratado como imutável.

## 3.2 Categorização canônica do NPS

A primeira variável derivada que vamos criar é a `categoria_nps`, que traduz o `nps_score` (0.0 a 10.0 em float) para os três buckets canônicos do NPS: detrator, neutro e promotor. Essa coluna é o que vamos usar como target na fase de classificação (notebook 06), e também serve para segmentar gráficos por bucket no notebook 04.

### Por que vetorizar essa transformação no módulo?

A função foi extraída para `src/nps/features.py` em vez de ficar inline no notebook. Três razões para essa escolha:

- **Reutilização:** a mesma classificação vai aparecer no notebook 04 (segmentar EDA por bucket), no 06 (treinar classificador) e nos slides. Manter em um único lugar evita divergência entre notebooks.
- **Performance:** a versão vetorizada usa `pd.cut`, que é nativo do pandas e roda em C por baixo dos panos. Bem mais rápido que aplicar a função escalar via `.apply()` em cada linha. Para 2.500 linhas a diferença é imperceptível, mas é a escolha certa para um projeto que pode escalar depois.
- **Testabilidade:** com a função em módulo, dá para escrever teste unitário simples (em `tests/test_features.py`) confirmando que 6.9 vai para detrator, 7.0 para neutro, e 9.0 para promotor. É a forma de garantir que mudanças futuras não quebram a regra canônica.

### Constantes nomeadas em vez de números soltos

No módulo, defini `CORTE_DETRATOR_NEUTRO = 7` e `CORTE_NEUTRO_PROMOTOR = 9` em vez de espalhar os números 7 e 9 pelo código. Isso é uma prática simples mas valiosa: se um dia a empresa decidir adotar uma régua diferente (alguns NPS internos usam 0-6 detrator e 7-10 promotor sem neutro, por exemplo), basta mudar a constante em um lugar.

In [2]:
from nps.features import adicionar_categoria_nps, CATEGORIAS_NPS_ORDEM

# Adiciona a coluna categoria_nps usando a funcao do modulo
dados = adicionar_categoria_nps(dados)

# Confirma o tipo da coluna criada
print("Tipo da coluna criada:", dados["categoria_nps"].dtype)
print("Ordenada?", dados["categoria_nps"].cat.ordered)
print("Categorias na ordem definida:", list(dados["categoria_nps"].cat.categories))
print()

# Confirma a distribuicao (mesmo resultado que vimos no notebook 02)
print("Distribuicao da nova coluna:")
print(dados["categoria_nps"].value_counts().reindex(list(CATEGORIAS_NPS_ORDEM)))

Tipo da coluna criada: category
Ordenada? True
Categorias na ordem definida: ['detrator', 'neutro', 'promotor']

Distribuicao da nova coluna:
categoria_nps
detrator    2109
neutro       281
promotor     110
Name: count, dtype: int64


A coluna foi criada com sucesso, do tipo `category` ordenada (detrator < neutro < promotor). A distribuição (2109 / 281 / 110) bate exatamente com o que vimos no notebook 02, confirmando que a categorização do módulo segue a regra canônica.

O dataset agora tem 20 colunas (19 originais + a nova `categoria_nps`). O `nps_score` continua intacto na sua representação contínua, então temos as duas formas disponíveis para as próximas fases: regressão na escala 0 a 10 e classificação nos três buckets.

In [3]:
# Sanidade: o dataset tem mais uma coluna agora
print(f"Shape atual: {dados.shape}")
print(f"Colunas adicionadas em relacao ao bruto: {[c for c in dados.columns if c not in carregar_dataset_bruto().columns]}")

Shape atual: (2500, 20)
Colunas adicionadas em relacao ao bruto: ['categoria_nps']


A categorização está pronta. As próximas seções vão tratar das outras transformações (categorical para `customer_region`, avaliação de log para variáveis assimétricas, etc) antes de salvar o dataset processado.